# NEXUS LLM — Module 5: Generation Lab

Explore how a causal language model turns a prompt into generated text.

This notebook covers greedy decoding, sampling, temperature, Top-K, Top-P, repetition controls, stopping criteria, KV cache, strategy comparison, and a reusable `generate_text()` function.

Run cells from top to bottom. This is inference with a pretrained model, not model training.

## 1. Imports and configuration

In [1]:
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilbert/distilgpt2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


## 2. Load tokenizer and model

`AutoModelForCausalLM` includes the language-modeling head. `model.eval()` sets inference behavior; `torch.inference_mode()` disables gradient tracking.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.to(DEVICE)
model.eval()

print("Model loaded")
print("Vocabulary size:", model.config.vocab_size)
print("Maximum positions:", model.config.n_positions)
print("EOS:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token, tokenizer.pad_token_id)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded
Vocabulary size: 50257
Maximum positions: 1024
EOS: <|endoftext|> 50256
PAD: <|endoftext|> 50256


## 3. Prepare a prompt

In [3]:
prompt = "Artificial intelligence is transforming"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
prompt_ids = inputs["input_ids"]
prompt_tokens = tokenizer.convert_ids_to_tokens(prompt_ids[0].tolist())

print("Prompt:", prompt)
print("Tokens:", prompt_tokens)
print("Input shape:", tuple(prompt_ids.shape))

Prompt: Artificial intelligence is transforming
Tokens: ['Art', 'ificial', 'Ġintelligence', 'Ġis', 'Ġtransforming']
Input shape: (1, 5)


## 4. Greedy generation

`do_sample=False` chooses the highest-scoring token at each step. `max_new_tokens` limits only the newly generated tokens.

In [4]:
start = time.perf_counter()
with torch.inference_mode():
    greedy_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True
    )
elapsed = time.perf_counter() - start

greedy_text = tokenizer.decode(greedy_ids[0], skip_special_tokens=True)
print(greedy_text)
print("New tokens:", greedy_ids.shape[1] - prompt_ids.shape[1])
print(f"Elapsed: {elapsed:.2f} seconds")

Artificial intelligence is transforming the way we think about our lives.










































New tokens: 50
Elapsed: 1.22 seconds


## 5. Sampling with temperature

`do_sample=True` samples from a probability distribution. Lower temperature makes it more concentrated; higher temperature makes it flatter.

In [5]:
torch.manual_seed(42)
with torch.inference_mode():
    temperature_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True
    )
print(tokenizer.decode(temperature_ids[0], skip_special_tokens=True))

Artificial intelligence is transforming the lives of a vast swathe of the world. And the technology is growing and rapidly gaining popularity.































## 6. Top-K and Top-P sampling

Top-K retains a fixed maximum number of candidates. Top-P retains a variable-sized group based on cumulative probability.

In [6]:
torch.manual_seed(42)
with torch.inference_mode():
    top_k_p_ids = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True
    )
print(tokenizer.decode(top_k_p_ids[0], skip_special_tokens=True))

Artificial intelligence is transforming the lives of a vast swathe of the world. And the tech industry has already been playing a role in that.


## 7. Repetition controls

`repetition_penalty` discourages reuse of previously seen tokens. `no_repeat_ngram_size=3` prevents an identical three-token sequence from appearing again. Very strong controls can make text unnatural.

In [7]:
torch.manual_seed(42)
with torch.inference_mode():
    controlled_ids = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True
    )
print(tokenizer.decode(controlled_ids[0], skip_special_tokens=True))

Artificial intelligence is transforming the lives of a vast swathe of people, including Americans.
The growing body of research on artificial intelligence has revealed that it can alter our perceptions about what's going on in society and beyond – at least for some time - without fear of consequences or reprisal from those we might have imagined (


## 8. Compare generation strategies

In [8]:
strategies = {
    "Greedy": {"do_sample": False},
    "Temperature": {"do_sample": True, "temperature": 0.7},
    "Top-K": {"do_sample": True, "temperature": 0.8, "top_k": 50},
    "Top-P": {"do_sample": True, "temperature": 0.8, "top_p": 0.9},
    "Top-K + Top-P": {
        "do_sample": True, "temperature": 0.8,
        "top_k": 50, "top_p": 0.9
    }
}

comparison_rows = []
for name, settings in strategies.items():
    torch.manual_seed(42)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
            **settings
        )

    new_ids = output_ids[0, prompt_ids.shape[1]:]
    comparison_rows.append({
        "Strategy": name,
        "New tokens": int(new_ids.numel()),
        "Continuation": tokenizer.decode(new_ids, skip_special_tokens=True)
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,Strategy,New tokens,Continuation
0,Greedy,50,the way we think about our lives.\n\n\n\n\n\n...
1,Temperature,50,the lives of a vast swathe of the world. And ...
2,Top-K,50,the lives of a vast swathe of the world. And ...
3,Top-P,25,the lives of a vast swathe of the world. And ...
4,Top-K + Top-P,25,the lives of a vast swathe of the world. And ...
